# Gemma-Cyber v0.2 (exp-002): Free Cloud QLoRA Training & GGUF Export

This notebook fine-tunes **Gemma-3-4B-it** on the curated `sft_v0.2.jsonl` cybersecurity dataset using **QLoRA** (4-bit quantization + LoRA adapters), merges the weights, and exports a quantized **GGUF** model for local inference with Ollama.

`sft_v0.2` is the first dataset with genuine answer-level diversity (277/277 unique answers vs. v0.1's 91/360) across 15 task types, and explicitly teaches exact ATT&CK IDs (e.g. Kerberoasting = **T1558.003**, not the hallucinated T1060). This is **exp-002**, the project's first *real* fine-tune (`gemma3-cyber:v0.1` was only a system-prompt alias of the base model).

### Hardware Requirement
* Free Google Colab **T4 GPU** (15GB VRAM) or **L4/A100**. Sequence length 1024 keeps a T4 comfortable.

### Gemma-3 chat-format note
Gemma templates historically reject a standalone `system` role and use the role name `model` (not `assistant`). This notebook renders the training text with the repo's verified `to_gemma_chat_text()` (folds system into the first user turn) instead of relying on the tokenizer template, and masks the prompt so loss is computed only on the `model` turns.

## Step 1: Install Dependencies & Check GPU

In [ ]:
!nvidia-smi
!pip install -q torch transformers peft trl bitsandbytes accelerate datasets pyyaml

## Step 1.5: Authenticate with Hugging Face

`google/gemma-3-4b-it` is a **gated** model, so the download needs an authenticated HF token.

1. **One-time access grant:** open [huggingface.co/google/gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it) while logged in and click **"Agree and access repository"**. Wait until the page shows you have access (the Gemma gate can take minutes–hours).
2. **Create a READ token:** [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
3. **Store it in Colab:** click the 🔑 (Secrets) icon in the left sidebar → add a secret named `HF_TOKEN` with your token → enable **Notebook access**. (If you skip this, the cell below falls back to an interactive prompt.)

In [ ]:
# Authenticate so the gated Gemma download is authorized.
# Prefers a Colab secret named HF_TOKEN; falls back to an interactive prompt.
from huggingface_hub import login, whoami

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    pass

login(token=token)  # token=None -> interactive widget to paste your HF token
print('Authenticated as:', whoami()['name'])


## Step 2: Clone Gemma4-CyberAi Repository & Validate Dataset

In [ ]:
import json, sys
from pathlib import Path

# If running in Colab without git clone, clone the repo (also puts src/ on the path).
dataset_path = 'data/training/sft_v0.2.jsonl'
if not Path(dataset_path).exists():
    print('Cloning repository...')
    !git clone https://github.com/novrusshehaj/Gemma4-CyberAi.git
    %cd Gemma4-CyberAi

# Make the package importable so we can reuse the VERIFIED Gemma formatter.
sys.path.insert(0, str(Path('src').resolve()))

# Verify dataset
with open(dataset_path, 'r', encoding='utf-8') as f:
    items = [json.loads(line) for line in f if line.strip()]
print(f'Successfully loaded {len(items)} training examples from {dataset_path}.')
assert len({json.dumps(it["id"]) for it in items}) == len(items), 'duplicate ids!'

## Step 3: Load Base Model with 4-bit Quantization (QLoRA)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = 'google/gemma-3-4b-it'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 4: Train LoRA Adapter with SFTTrainer

In [ ]:
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

# Render each example to Gemma-3 turn format with the repo's VERIFIED formatter
# (folds the `system` role into the first user turn; emits `model` turns). This avoids
# depending on the tokenizer chat template, which historically rejects a system role.
from gemma_cyber.data.formatting import to_gemma_chat_text

formatted = [{'text': to_gemma_chat_text(it['messages'])} for it in items]
train_dataset = Dataset.from_list(formatted)
print('Sample formatted example:\n', formatted[0]['text'][:300], '...')

# Mask the prompt: compute loss ONLY on the assistant/model completion. The response
# template marks where the model turn begins.
response_template = '<start_of_turn>model\n'
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir='./results_gemma3_cyber_v0.2',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    optim='paged_adamw_8bit',
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    seed=42,
    report_to='none'
)

# NOTE (TRL version drift): newer TRL moves max_seq_length/dataset_text_field onto SFTConfig.
# If SFTTrainer(...) raises a TypeError on these kwargs, switch to trl.SFTConfig accordingly.
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    dataset_text_field='text',
    max_seq_length=1024,
    data_collator=collator,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()
trainer.model.save_pretrained('./final_adapter')
tokenizer.save_pretrained('./final_adapter')
print('Training complete! Adapter saved to ./final_adapter')

## Step 5: Merge LoRA Adapter & Export to GGUF

In [ ]:
# Merge LoRA adapter into base model in FP16
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map='cpu',
    trust_remote_code=True
)
merged_model = PeftModel.from_pretrained(base_model, './final_adapter')
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained('./gemma3-cyber-v0.2-merged')
tokenizer.save_pretrained('./gemma3-cyber-v0.2-merged')
print('Merged model saved.')

# Clone llama.cpp and convert to GGUF
!git clone https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py ./gemma3-cyber-v0.2-merged --outfile gemma3-cyber-v0.2.gguf --outtype f16

# Quantize to Q4_K_M for local Ollama serving
!cd llama.cpp && make llama-quantize
!./llama.cpp/llama-quantize gemma3-cyber-v0.2.gguf gemma3-cyber-v0.2-Q4_K_M.gguf Q4_K_M
print('Exported gemma3-cyber-v0.2-Q4_K_M.gguf ready for Ollama!')

## Step 6: Create Ollama Model Locally & Evaluate

Download `gemma3-cyber-v0.2-Q4_K_M.gguf` to your local machine and run:
```bash
ollama create gemma3-cyber:v0.2 -f Modelfile.template   # point the FROM line at the v0.2 GGUF
ollama run gemma3-cyber:v0.2 "Explain the MITRE ATT&CK technique for Kerberoasting."
```

Then run the pre-registered evaluation (base vs. v0.2) on the frozen v2 anchor **and** the
new targeted v3 instrument, scoring the held-out `test` split only for the final comparison:
```bash
# do-no-harm / regression anchor (v2)
python scripts/run_baseline.py --model gemma3-cyber:v0.2 \
    --benchmark data/evaluation/benchmark_v2.jsonl --split test \
    --out experiments/exp-002-gemma3-cyber-v0.2/v2-test
# targeted sensitivity instrument (v3) — ATT&CK precision, false premises, factual scorer
python scripts/run_baseline.py --model gemma3-cyber:v0.2 \
    --benchmark data/evaluation/benchmark_v3.jsonl --split test \
    --out experiments/exp-002-gemma3-cyber-v0.2/v3-test
```
Compare against the base `gemma3:4b` run on the same benchmarks/splits. See
`docs/experiments/exp-002.md` for the full hypothesis, control/treatment, and success
criteria (v2 do-no-harm guard + v3 hallucination/ATT&CK precision targets).